In [ ]:
!nvidia-smi

Wed May 28 22:25:22 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8              9W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install --quiet transformers pytorch-lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.1/823.1 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 97.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 37.1 MB/s eta 0:00:00


In [ ]:
!pip install torch pytorch-lightning transformers pandas scikit-learn

In [ ]:
import json
import pandas as pd
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from sklearn.model_selection import train_test_split
from termcolor import colored
import textwrap

from torch.optim import AdamW

from transformers import (
    T5ForConditionalGeneration,
    T5TokenizerFast as T5Tokenizer,
)

In [ ]:
import json
from torch.utils.data import Dataset
import torch
from typing import List, Dict, Tuple
from transformers import T5Tokenizer, T5ForConditionalGeneration
import re
from collections import Counter

In [ ]:
with Path("/content/conversation.json").open() as json_file:
    conversation_data = json.load(json_file)


This function reads a call center conversation and extracts useful question-answer examples. These examples are used to train the AI model to better understand customer interactions. We organize the data into four categories: direct questions from clients, profile-based questions about the client, context-aware questions, and responses adapted to emotional or situational context


Direct QA (_extract_direct_qa) : We detect when a client asks something and the agent replies. These pairs are extracted to teach the AI how to answer

Profile QA (via _extract_profile_qa)
"We use the customer profile (like their name, contract type, or provider) to create questions and answers. This helps the AI give more personalized answ


 cleans a piece of text by removing extra spaces and speaker labels like "Agent:" or "Client:" at the beginning.

checks if a question-answer pair is good enough to be used.
It makes sure both question and answer are not too short and that the answer is not too generic (like "yes" or "ok").



In [ ]:
class T5QADataProcessor:
    """
    Improved processor for T5 training on call center conversations
    Focused specifically on QA tasks with better data quality
    """

    def __init__(self, model_name="t5-small"):
        self.tokenizer = T5Tokenizer.from_pretrained(model_name)
        self.model = T5ForConditionalGeneration.from_pretrained(model_name)

    def extract_training_data(self, conversation_data: Dict) -> List[Dict]:
        """
        Extract and format data for T5 QA training with improved quality
        """
        training_examples = []

        try:
            # 1. Direct QA pairs - IMPROVED
            qa_pairs = self._extract_direct_qa(conversation_data)
            for qa in qa_pairs:
                training_examples.append({
                    'input_text': f"question: {qa['question']}",
                    'target_text': qa['answer'],
                    'task_type': 'direct_qa'
                })

            # 2. Profile-based QA - ENHANCED
            profile_qa = self._extract_profile_qa(conversation_data)
            for qa in profile_qa:
                training_examples.append({
                    'input_text': f"question: {qa['question']}",
                    'target_text': qa['answer'],
                    'task_type': 'profile_qa'
                })

            # 3. Context-aware QA - NEW
            context_qa = self._extract_context_qa(conversation_data)
            for qa in context_qa:
                training_examples.append({
                    'input_text': f"question: {qa['question']} context: {qa['context']}",
                    'target_text': qa['answer'],
                    'task_type': 'context_qa'
                })

            # 4. Situation-based responses - NEW
            situation_qa = self._extract_situation_responses(conversation_data)
            training_examples.extend(situation_qa)

        except Exception as e:
            print(f"Error extracting data: {e}")
            return []

        return self._filter_and_clean_examples(training_examples)

    def _extract_direct_qa(self, data: Dict) -> List[Dict]:
        """
        IMPROVED: Extract high-quality agent responses to client inputs
        """
        conversation = data.get('conversation', [])
        qa_pairs = []

        for i in range(len(conversation) - 1):
            current = conversation[i]
            next_msg = conversation[i + 1]

            # Client question -> Agent response
            if (current['speaker'] == 'client' and
                next_msg['speaker'] == 'agent'):

                # Clean and validate the pair
                question = self._clean_text(current['text'])
                answer = self._clean_text(next_msg['text'])

                if self._is_valid_qa_pair(question, answer):
                    qa_pairs.append({
                        'question': question,
                        'answer': answer
                    })

        return qa_pairs

    def _extract_profile_qa(self, data: Dict) -> List[Dict]:
        """
        ENHANCED: Extract profile-based Q&A with better question formulation
        """
        profile_qa = []
        profile = data.get('profile', {})

        # Client name
        if profile.get('name'):
            profile_qa.append({
                'question': "Quel est le nom du client?",
                'answer': profile['name']
            })

        # Process history items with improved parsing
        history_mappings = {
            'Fournisseur actuel': [
                "Qui est le fournisseur d'énergie du client?",
                "Quel est le fournisseur actuel?"
            ],
            'Type de contrat': [
                "Quel type de contrat a le client?",
                "Quelle est la formule tarifaire?"
            ],
            'Double usage': [
                "Quel est l'usage du contrat?",
                "Le contrat est-il à usage mixte?"
            ],
            'Décompte annuel': [
                "Quand le client reçoit-il son décompte?",
                "À quelle période arrive le décompte annuel?"
            ]
        }

        for history_item in profile.get('history', []):
            for key, questions in history_mappings.items():
                if key in history_item:
                    try:
                        answer = history_item.split(':', 1)[1].strip()
                        # Generate multiple questions for same answer
                        for question in questions:
                            profile_qa.append({
                                'question': question,
                                'answer': answer
                            })
                    except (IndexError, AttributeError):
                        continue

        return profile_qa

    # def _extract_context_qa(self, data: Dict) -> List[Dict]:
    #     """
    #     NEW: Extract context-aware QA pairs
    #     """
    #     context_qa = []
    #     conversation = data.get('conversation', [])

    #     # Build context from conversation flow
    #     context_info = self._build_context(data)

    #     # Generate context-aware questions
    #     context_questions = [
    #         {
    #             'question': "Le client est-il satisfait de son fournisseur actuel?",
    #             'context': context_info,
    #             'answer': self._determine_satisfaction(conversation)
    #         },
    #         {
    #             'question': "Quelle est la préoccupation principale du client?",
    #             'context': context_info,
    #             'answer': self._extract_main_concern(conversation)
    #         },
    #         {
    #             'question': "Le client est-il ouvert au changement?",
    #             'context': context_info,
    #             'answer': self._determine_openness_to_change(conversation)
    #         }
    #     ]

    #     return [qa for qa in context_questions if qa['answer']]

    # def _extract_situation_responses(self, data: Dict) -> List[Dict]:
    #     """
    #     NEW: Extract situation-based response patterns
    #     """
    #     situation_responses = []
    #     emotion = data.get('label_emotion', '')
    #     conversation = data.get('conversation', [])

    #     # Emotion-based responses
    #     if emotion == 'frustrated':
    #         situation_responses.append({
    #             'input_text': "question: Comment répondre à un client frustré?",
    #             'target_text': "Je comprends votre frustration. Laissez-moi vous expliquer clairement les options disponibles.",
    #             'task_type': 'situation_response'
    #         })
    #     elif emotion == 'neutral':
    #         situation_responses.append({
    #             'input_text': "question: Comment approcher un client neutre?",
    #             'target_text': "Permettez-moi de vous présenter les avantages de notre offre.",
    #             'task_type': 'situation_response'
    #         })

    #     # Situation-specific responses based on conversation patterns
    #     if self._has_pricing_concerns(conversation):
    #         situation_responses.append({
    #             'input_text': "question: Comment répondre aux préoccupations tarifaires?",
    #             'target_text': "Je comprends vos préoccupations sur les tarifs. Avec un contrat fixe, vous seriez protégé des hausses pendant un an.",
    #             'task_type': 'situation_response'
    #         })

    #     return situation_responses

    def _clean_text(self, text: str) -> str:
        """Clean and normalize text"""
        if not text:
            return ""

        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text.strip())

        # Remove speaker labels if present
        text = re.sub(r'^(Agent|Client):\s*', '', text)

        return text

    def _is_valid_qa_pair(self, question: str, answer: str) -> bool:
        """Validate QA pair quality"""
        if not question or not answer:
            return False

        # Minimum length requirements
        if len(question.split()) < 2 or len(answer.split()) < 3:
            return False

        # Avoid very generic responses
        generic_responses = [
            "oui", "non", "d'accord", "ok", "très bien", "pourquoi pas"
        ]
        if answer.lower().strip() in generic_responses:
            return False

        return True

    def _build_context(self, data: Dict) -> str:
        """Build context string from conversation data"""
        context_parts = []

        profile = data.get('profile', {})
        if profile.get('name'):
            context_parts.append(f"Client: {profile['name']}")

        # Add key profile information
        for history_item in profile.get('history', []):
            context_parts.append(history_item)

        return " | ".join(context_parts)

    # def _determine_satisfaction(self, conversation: List[Dict]) -> str:
    #     """Determine client satisfaction from conversation"""
    #     client_messages = [msg['text'].lower() for msg in conversation if msg['speaker'] == 'client']
    #     full_text = " ".join(client_messages)

    #     if any(word in full_text for word in ['frustré', 'marre', 'énerve', 'arnaqué', 'ras-le-bol']):
    #         return "Le client exprime de la frustration avec son fournisseur actuel"
    #     elif any(word in full_text for word in ['satisfait', 'content', 'bien']):
    #         return "Le client semble satisfait"
    #     else:
    #         return "Satisfaction neutre"

    # def _extract_main_concern(self, conversation: List[Dict]) -> str:
    #     """Extract main client concern"""
    #     client_messages = [msg['text'].lower() for msg in conversation if msg['speaker'] == 'client']
    #     full_text = " ".join(client_messages)

    #     if any(word in full_text for word in ['prix', 'tarif', 'facture', 'paye', 'coût']):
    #         return "Préoccupations tarifaires et coûts énergétiques"
    #     elif any(word in full_text for word in ['changement', 'changer', 'fournisseur']):
    #         return "Hésitation concernant le changement de fournisseur"
    #     elif any(word in full_text for word in ['comprends pas', 'compliqué', 'difficile']):
    #         return "Besoin d'explications et de clarifications"
    #     else:
    #         return ""

    # def _determine_openness_to_change(self, conversation: List[Dict]) -> str:
    #     """Determine client's openness to change"""
    #     client_messages = [msg['text'].lower() for msg in conversation if msg['speaker'] == 'client']
    #     full_text = " ".join(client_messages)

    #     if any(word in full_text for word in ['oui', 'pourquoi pas', 'd\'accord', 'intéressé']):
    #         return "Client ouvert au changement"
    #     elif any(word in full_text for word in ['non', 'ne veux plus', 'marre de changer']):
    #         return "Client réticent au changement"
    #     else:
    #         return "Position neutre"

    # def _has_pricing_concerns(self, conversation: List[Dict]) -> bool:
    #     """Check if conversation involves pricing concerns"""
    #     client_messages = [msg['text'].lower() for msg in conversation if msg['speaker'] == 'client']
    #     full_text = " ".join(client_messages)
    #     return any(word in full_text for word in ['prix', 'tarif', 'facture', 'paye', 'coût', 'cher'])

    def _filter_and_clean_examples(self, examples: List[Dict]) -> List[Dict]:
        """Filter and clean training examples"""
        cleaned_examples = []

        for example in examples:
            # Clean input and target text
            input_text = self._clean_text(example['input_text'])
            target_text = self._clean_text(example['target_text'])

            # Skip empty or too short examples
            if (len(input_text.split()) < 3 or
                len(target_text.split()) < 2):
                continue

            cleaned_examples.append({
                'input_text': input_text,
                'target_text': target_text,
                'task_type': example['task_type']
            })

        return cleaned_examples

First, it processes all conversations using our T5QADataProcessor

It collects all the question-answer pairs from every conversation

Dataset Balancing:

It ensures we have enough examples for each task type (minimum 5 examples)"
"If a task type has too few examples, it warns us about potential problems


Train/Test Split:

It divides the data into two parts: 80% for training the AI, 20% for testing how well it learned
"It uses 'stratified splitting' to ensure both training and test sets have examples from all task types"





In [ ]:
def create_balanced_training_dataset(conversation_data, tokenizer, test_split=0.2):
    """
    Create balanced training dataset with improved quality control
    """
    processor = T5QADataProcessor()
    all_training_data = []

    for conv in conversation_data:
        training_examples = processor.extract_training_data(conv)
        all_training_data.extend(training_examples)

    print(f"Total training examples: {len(all_training_data)}")

    # Balance dataset by task type
    task_counts = {}
    for item in all_training_data:
        task_type = item['task_type']
        task_counts[task_type] = task_counts.get(task_type, 0) + 1

    print("\nTask distribution (before balancing):")
    for task, count in task_counts.items():
        print(f"  {task}: {count}")

    # Ensure minimum examples per task
    min_examples = 5
    balanced_data = []

    for task_type in task_counts:
        task_examples = [item for item in all_training_data if item['task_type'] == task_type]

        if len(task_examples) >= min_examples:
            balanced_data.extend(task_examples)
        else:
            print(f"Warning: Only {len(task_examples)} examples for {task_type}")
            balanced_data.extend(task_examples)

    # Train/test split
    from sklearn.model_selection import train_test_split

    if len(set(item['task_type'] for item in balanced_data)) > 1:
        train_data, test_data = train_test_split(
            balanced_data,
            test_size=test_split,
            random_state=42,
            stratify=[item['task_type'] for item in balanced_data]
        )
    else:
        train_data, test_data = train_test_split(
            balanced_data,
            test_size=test_split,
            random_state=42
        )

    print(f"\nFinal dataset sizes:")
    print(f"Training: {len(train_data)} samples")
    print(f"Test: {len(test_data)} samples")

    return train_data, test_data

In [ ]:
def generate_answer(model, tokenizer, input_text):
    """
    Generate clean, contextual answers for call center QA
    """
    model.eval()

    # Ensure proper task prefix
    if not input_text.startswith("question:"):
        input_text = f"question: {input_text}"

    encoding = tokenizer(
        input_text,
        truncation=True,
        max_length=512,
        padding=False,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=encoding["input_ids"],
            attention_mask=encoding["attention_mask"],
            num_beams=4,
            max_length=128,
            repetition_penalty=1.3,
            length_penalty=1.0,
            no_repeat_ngram_size=2,
            early_stopping=True,
            do_sample=False,
            temperature=0.7
        )

    response = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    # Clean response
    response = re.sub(r'\b(Client|Agent):\s*', '', response)
    response = re.sub(r'^(question:|answer:)\s*', '', response, flags=re.IGNORECASE)
    response = response.strip()

    return response


In [ ]:
processor = T5QADataProcessor()


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:
examples = processor.extract_training_data(conversation_data[1])
for ex in examples:
    print(ex)


{'input_text': 'question: Je suis chez Enico depuis plus d’un an, je ne sais pas exactement.', 'target_text': 'Avez-vous le gaz et l’électricité ?', 'task_type': 'direct_qa'}
{'input_text': 'question: Oui, j’ai les deux.', 'target_text': 'Combien payez-vous ?', 'task_type': 'direct_qa'}
{'input_text': 'question: Ils m’ont remboursé 100€, mais maintenant je paye 260€. Avant c’était 200€.', 'target_text': 'C’est sûrement lié à une formule variable.', 'task_type': 'direct_qa'}
{'input_text': 'question: Oui, mais j’ai été chez Lumineuse avant et j’ai payé plus de 1000€. J’ai un seul radiateur !', 'target_text': 'Vous êtes peut-être en tarif variable, c’est ce qui cause ces hausses.', 'task_type': 'direct_qa'}
{'input_text': 'question: Je ne veux plus changer, on finit toujours par payer plus. J’en ai marre de tout ça.', 'target_text': 'Comprenez, avec un tarif fixe, vous seriez protégée pendant 1 an.', 'task_type': 'direct_qa'}
{'input_text': 'question: Peut-être. Mais j’ai déjà donné, mai

**#  TRAINING MODEL**

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


**Split the data**

In [ ]:
train_data, test_data = create_balanced_training_dataset(conversation_data, tokenizer)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Total training examples: 34

Task distribution (before balancing):
  direct_qa: 17
  profile_qa: 8
  context_qa: 6
  situation_response: 3

Final dataset sizes:
Training: 27 samples
Test: 7 samples


**Convert to HuggingFace Dataset**

In [ ]:
from datasets import Dataset as HFDataset


***Converts questions into numerical tokens like translating words into numbers***

In [ ]:
def preprocess_qa_data(examples):

    # Tokenize inputs (your input_text already contains the full prompt)
    model_inputs = tokenizer(
        examples['input_text'],
        max_length=512,
        truncation=True,
        padding=False  # Will be handled by data collator
    )

    # Tokenize targets
    labels = tokenizer(
        examples['target_text'],
        max_length=128,
        truncation=True,
        padding=False
    )

    # Add labels to model inputs
    #Links each question with its correct answer for training
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


**Keeps only the question-answer tasks we want to focus on**

In [ ]:
qa_task_types = ['direct_qa', 'profile_qa', 'context_qa']
train_data = [ex for ex in train_data if ex['task_type'] in qa_task_types]
test_data = [ex for ex in test_data if ex['task_type'] in qa_task_types]

In [ ]:
train_dataset = HFDataset.from_list(train_data)
test_dataset = HFDataset.from_list(test_data)

In [ ]:
train_dataset = train_dataset.map(
    preprocess_qa_data,
    batched=True,
    remove_columns=['input_text', 'target_text', 'task_type']
)

test_dataset = test_dataset.map(
    preprocess_qa_data,
    batched=True,
    remove_columns=['input_text', 'target_text', 'task_type']
)



Map:   0%|          | 0/27 [00:00<?, ? examples/s]

Map:   0%|          | 0/7 [00:00<?, ? examples/s]

In [ ]:
#Load model
model = T5ForConditionalGeneration.from_pretrained("t5-small")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


**Training configuration**


In [ ]:
# import evaluate

# metric = evaluate.load("squad")

# def compute_metrics(eval_preds):
#     preds, labels = eval_preds

#     # If preds is a tuple (from generate), get the first element
#     if isinstance(preds, tuple):
#         preds = preds[0]

#     # Convert tensors to lists if needed
#     if hasattr(preds, "tolist"):
#         preds = preds.tolist()
#     if hasattr(labels, "tolist"):
#         labels = labels.tolist()

#     # Replace -100 in labels as tokenizer.pad_token_id for decoding
#     labels = [
#         [token if token != -100 else tokenizer.pad_token_id for token in label]
#         for label in labels
#     ]

#     decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
#     decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

#     result = metric.compute(predictions=decoded_preds, references=decoded_labels)
#     return {
#         "exact_match": result["exact_match"],
#         "f1": result["f1"],
#     }





In [ ]:
# !pip install --upgrade transformers


In [ ]:
!pip uninstall transformers -y
!pip install transformers==4.37.2


Found existing installation: transformers 4.37.2
Uninstalling transformers-4.37.2:
  Successfully uninstalled transformers-4.37.2
  Using cached transformers-4.37.2-py3-none-any.whl.metadata (129 kB)
Using cached transformers-4.37.2-py3-none-any.whl (8.4 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 4.1.0 requires transformers<5.0.0,>=4.41.0, but you have transformers 4.37.2 which is incompatible.


In [ ]:
# import transformers
# print(transformers.__version__)  #  4.36.2


In [ ]:
!pip install peft==0.10.0


In [ ]:
training_args = TrainingArguments(
    output_dir="./t5_qa_output",

    # Training schedule
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,

    # Learning rate and optimization
    learning_rate=3e-4,
    warmup_steps=1000,
    weight_decay=0.01,
    adam_epsilon=1e-8,
    max_grad_norm=1.0,

    # Learning rate scheduling
    lr_scheduler_type="linear",
    warmup_ratio=0.1,

    # Evaluation and logging
    evaluation_strategy="steps",
    eval_steps=500,
    logging_dir="./logs",
    logging_steps=100,
    logging_strategy="steps",

    # Saving strategy
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Performance optimizations
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=4,
    remove_unused_columns=False,

    # Reproducibility
    seed=42,
    data_seed=42,


    # Hub and reporting
    push_to_hub=False,
    report_to="tensorboard",

    # Early stopping (optional)
    # early_stopping_patience=3,
    # early_stopping_threshold=0.001,
)


In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    return_tensors="pt"
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)


trainer.train()

trainer.save_model("./t5_qa_final_model")

/usr/local/lib/python3.11/dist-packages/accelerate/accelerator.py:446: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False)
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoade

Step,Training Loss,Validation Loss


In [ ]:
from transformers import Trainer, TrainingArguments


In [ ]:
# !pip install --upgrade accelerate


In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)


In [ ]:
# !pip install accelerate==0.30.0

**# Sample questions for quick testing**

In [ ]:
def test_model_on_real_dataset(model, tokenizer, dataset_path, processor=None, max_examples=5):
    """
    Test the trained T5 model on real call center data.
    - model: Trained T5 model (T5ForConditionalGeneration)
    - tokenizer: Corresponding T5 tokenizer
    - dataset_path: Path to the JSON file (list of conversations)
    - processor: Instance of T5QADataProcessor (if None, will create one)
    - max_examples: Number of examples to print per conversation
    """
    import json

    # Load dataset
    with open(dataset_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if processor is None:
        processor = T5QADataProcessor()

    for i, conv in enumerate(data):
        print(f"\n=== Conversation {i+1} (client_id: {conv.get('client_id', 'N/A')}) ===")
        qa_examples = processor.extract_training_data(conv)
        count = 0
        for ex in qa_examples:
            if ex['task_type'] not in ['direct_qa', 'profile_qa', 'context_qa']:
                continue  # Only test QA tasks
            input_text = ex['input_text']
            target_text = ex['target_text']
            # Generate answer
            model.eval()
            encoding = tokenizer(
                input_text,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            ).to(model.device)
            with torch.no_grad():
                generated_ids = model.generate(
                    input_ids=encoding["input_ids"],
                    attention_mask=encoding["attention_mask"],
                    num_beams=4,
                    max_length=128,
                    repetition_penalty=1.3,
                    length_penalty=1.0,
                    no_repeat_ngram_size=2,
                    early_stopping=True,
                    do_sample=False,
                    temperature=0.7
                )
            response = tokenizer.decode(generated_ids[0], skip_special_tokens=True).strip()
            print(f"\nQ: {input_text}")
            print(f"Expected: {target_text}")
            print(f"Model:    {response}")
            count += 1
            if count >= max_examples:
                break
        print("-" * 60)

In [ ]:
test_model_on_real_dataset(
    model=model,
    tokenizer=tokenizer,
    dataset_path="/content/conversation.json",
    processor=processor,
    max_examples=5
)


=== Conversation 1 (client_id: C002) ===


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:392: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(



Q: question: Le fournisseur ? Qu’est-ce qu’il a fait ?
Expected: Vous êtes toujours chez le même fournisseur ?
Model:    Qu’est-ce qu’il a fait

Q: question: Oui, c’est le label.
Expected: Depuis combien de temps ? Avez-vous eu une révision tarifaire ?
Model:    c’est le label

Q: question: Non, non, pas de révision.
Expected: Vous avez un tarif social ?
Model:    pas de révision

Q: question: Non, non.
Expected: Vous recevez votre décompte à quel moment ?
Model:    Non

Q: question: Début mai.
Expected: Est-ce qu’on peut vous rappeler à ce moment-là pour faire une révision ?
Model:    Début mai.
------------------------------------------------------------

=== Conversation 2 (client_id: C003) ===

Q: question: Je suis chez Enico depuis plus d’un an, je ne sais pas exactement.
Expected: Avez-vous le gaz et l’électricité ?
Model:    Je ne sais pas exactement

Q: question: Oui, j’ai les deux.
Expected: Combien payez-vous ?
Model:    j’ai les deux

Q: question: Ils m’ont remboursé 100€, 